# ✦ LILY WAN 2.2 — Adaptive Kaggle Studio — v6

This build fixes the v5 generated-code newline syntax bug before execution. It still prefers T4 x2 and automatically falls back to the P100 profile if Kaggle overrides the accelerator.


In [ ]:
import json, urllib.request

print('✦ Lily Wan 2.2 Studio — startup v6')
print('✓ Loading the last adaptive build and applying the syntax repair...')

V5_URL = 'https://raw.githubusercontent.com/benruiz1024-ops/hi/7268f44cfd9f610b156a08cc3d064b3427b60c63/LILY_WAN22_DUAL_T4_STUDIO.ipynb'
with urllib.request.urlopen(V5_URL, timeout=60) as r:
    nb = json.loads(r.read().decode('utf-8'))

cells = [c for c in nb['cells'] if c.get('cell_type') == 'code']
if not cells:
    raise RuntimeError('Could not load the v5 studio code cell.')
wrapper = ''.join(cells[0]['source'])

# v5 accidentally generated a quoted newline that became a literal line break.
# Replace that one source line structurally, so no escape matching is required.
fixed_lines = []
repaired = False
for line in wrapper.splitlines():
    if '_req_safe.write_text(' in line:
        indent = line[:len(line) - len(line.lstrip())]
        fixed_lines.append(indent + '_req_safe.write_text(chr(10).join(_lines) + chr(10))')
        repaired = True
    else:
        fixed_lines.append(line)
wrapper = chr(10).join(fixed_lines) + chr(10)

if not repaired:
    raise RuntimeError('Could not locate the v5 newline bug to repair.')

# Compile the repaired wrapper before doing any expensive setup.
compiled = compile(wrapper, 'LILY_WAN22_V6_REPAIRED_WRAPPER', 'exec')
print('✓ Syntax repair applied')
print('✓ Wrapper passed compile preflight')
print('✓ Starting adaptive Wan 2.2 setup...')
exec(compiled, globals(), globals())
